In [1]:
!pip install mne

In [2]:
!pip install mne-icalabel

In [3]:
!pip install torch

In [ ]:
import mne
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
from mne_icalabel import label_components

In [ ]:
file_path = "D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/sub-01_ses-01_task-innerspeech_eeg.bdf"

raw = mne.io.read_raw_bdf(file_path, preload=True, verbose="WARNING")
raw

In [ ]:
channels = raw.info['ch_names']
channels

In [ ]:
# BioSemi 128 montage 불러오기
montage = mne.channels.make_standard_montage('biosemi128')

# montage 적용
raw.set_montage(montage, on_missing='ignore')  # EXG 채널은 위치 정보 없이 유지하고, EEG 채널만 montage를 적용

In [ ]:
# montage 정보 출력
print(raw.get_montage())

In [ ]:
print(raw.info['dig'])     # 빈 리스트([])면 안됨

In [ ]:
events = mne.find_events(raw, shortest_event=1, verbose=True)

In [ ]:
print(events)

In [ ]:
# Re-referencing
raw.set_eeg_reference(ref_channels=['EXG1', 'EXG2'])

In [ ]:
eeg_d = raw.get_data()
eeg_d

In [ ]:
eeg_d.shape

In [ ]:
channel_names = raw.ch_names

# NumPy 배열을 DataFrame으로 변환
df = pd.DataFrame(eeg_d.T, columns=channel_names)
df

In [ ]:
raw.plot(n_channels=50, title='EEG Raw Data (first 50 channels)')

In [ ]:
# 0.5-100Hz Band-pass filter 적용
raw_bandpass = raw.filter(l_freq=0.5, h_freq=100)

In [ ]:
raw_bandpass.plot(n_channels=50, title='Filtered EEG Data (0.5-100 Hz)', show=True)

In [ ]:
psd = raw_bandpass.compute_psd(fmax=125)
psd.plot()

In [ ]:
# 50 Hz Notch-filter 적용
raw_notch = raw_bandpass.notch_filter(freqs=50)

In [ ]:
raw_notch.plot(n_channels=50, title='Filtered EEG Data (50 Hz)', show=True)

In [ ]:
psd = raw_notch.compute_psd(fmax=125)
psd.plot()

In [ ]:
# Sampling rate 출력
print(f"Sampling rate: {raw_notch.info['sfreq']} Hz")

In [ ]:
# Resampling
raw_resample = raw_notch.resample(sfreq=256)
raw_resample

In [ ]:
# Resampling 후 events 잘 변환되었는지 확인
events_resampled = mne.find_events(raw_resample, shortest_event=1)

print("Original events shape:", events.shape)
print("Resampled events shape:", events_resampled.shape)

print("Original events sample points (first 10):", events[:10,0])
print("Resampled events sample points (first 10):", events_resampled[:10,0])

In [ ]:
print(f"Sampling rate: {raw_resample.info['sfreq']} Hz")

In [ ]:
# Epoching
event_id = {'arriba': 31, 'abajo': 32, 'derecha': 33, 'izquierda': 34}

# EEG 채널만 선택한 epoch 생성
exg_names = ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8']
eeg_names = [ch for ch in raw_resample.ch_names if ch not in exg_names]

eeg = raw_resample.copy().pick_channels(eeg_names)

epochs = mne.Epochs(eeg, events, event_id=event_id, tmin=0, tmax=4.5,
                    baseline=None, preload=True, reject=None, flat=None)

In [ ]:
print(epochs._data.shape)

In [ ]:
# ICA
'''
ica = ICA(n_components=128, random_state=97, method='infomax', fit_params=dict(extended=True))
ica.fit(epochs)

# ICA 컴포넌트 자동 라벨링
ic_labels = label_components(epochs, ica, method='iclabel')  # eeg 또는 epochs 사용 가능

labels = ic_labels["labels"]

# 2. brain과 other 외 컴포넌트는 artifact로 판단해 제외
exclude_idx = [idx for idx, label in enumerate(labels) if label not in ['brain', 'other']]

ica.exclude = exclude_idx

# 3. 제외된 컴포넌트 시각화 (확인용)
ica.plot_components()

# 4. ICA 적용해 artifact 제거한 epoch 생성
clean_epochs = ica.apply(epochs.copy())

# 5. 결과 확인용 PSD 그리기
clean_epochs.compute_psd().plot()
'''

In [ ]:
# Epoch 개수 확인

#print(f"Total number of epochs: {len(clean_epochs)}")

In [ ]:
# clean_epochs.save('D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/sub-01_ses-01_eeg-epo.fif', overwrite=True)